In [2]:
## 1.Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp '/content/drive/My Drive/UAV/data1.zip' '/content/data1.zip'
!unzip '/content/data1.zip' -d '/content/'

Archive:  /content/data1.zip
   creating: /content/data/
   creating: /content/data/images/
   creating: /content/data/images/train/
  inflating: /content/data/images/train/1 (1).jpg  
  inflating: /content/data/images/train/1 (2).jpg  
  inflating: /content/data/images/train/1 (3).jpg  
  inflating: /content/data/images/train/1 (4).jpg  
  inflating: /content/data/images/train/1 (5).jpg  
  inflating: /content/data/images/train/1634016299438.jpg  
  inflating: /content/data/images/train/1634016299451.jpg  
  inflating: /content/data/images/train/1634016299471.jpg  
  inflating: /content/data/images/train/1634016299516.jpg  
  inflating: /content/data/images/train/1634016299526.jpg  
  inflating: /content/data/images/train/1634016299534.jpg  
  inflating: /content/data/images/train/1634016299544.jpg  
  inflating: /content/data/images/train/1634016299552.jpg  
  inflating: /content/data/images/train/1634016299562.jpg  
  inflating: /content/data/images/train/1634016299572.jpg  
  infla

In [4]:
pip install -U albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.4/606.4 kB 32.7 MB/s eta 0:00:00
  Attempting uninstall: albucore
    Found existing installation: albucore 0.0.19
    Uninstalling albucore-0.0.19:
      Successfully uninstalled albucore-0.0.19
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.20
    Uninstalling albumentations-1.4.20:
      Successfully uninstalled albumentations-1.4.20


In [5]:
!git clone https://github.com/ultralytics/ultralytics.git
!cd ultralytics && pip install .

Cloning into 'ultralytics'...
remote: Enumerating objects: 46509, done.
remote: Counting objects: 100% (702/702), done.
remote: Compressing objects: 100% (472/472), done.
remote: Total 46509 (delta 443), reused 410 (delta 228), pack-reused 45807 (from 1)
Receiving objects: 100% (46509/46509), 38.56 MiB | 21.52 MiB/s, done.
Resolving deltas: 100% (34611/34611), done.
Processing /content/ultralytics
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.36-py3-none-any.whl size=887317 sha256=643192b7eacd81fc64fa95ba1469fd26e39c069b8224bf449b5233cb852efc31
  Stored in directory: /tmp/pip-ephem-wheel-cache-e723in8q/wheels/9a/cd/d5/95912172899f8ec640166ff6eef49156b1b00d6b2ade4a3cb1
Successfully built ultralytics


In [7]:
import yaml
from ultralytics import YOLO
from google.colab import files, drive
import os

# Mount Google Drive to save weights
drive.mount('/content/drive')

# Paths to the unzipped dataset
train_images_path = '/content/data1/images/train'
val_images_path = '/content/data1/images/val'
train_labels_path = '/content/data1/labels/train'
val_labels_path = '/content/data1/labels/val'

# Create a temporary dictionary for data configuration
data_config = {
    'train': train_images_path,  # Path to training images
    'val': val_images_path,      # Path to validation images
    'nc': 35,                    # Number of classes
    'names': [                   # Class names
        "Traffic Signal", "Lamp Post", "Zebra Crossing", "Bike", "Car", "Rikshaw", "Tyre Works",
        "Tree", "Tractor", "Cattle", "Vegetation", "Electricity Pole", "Building", "Board", "Wall",
        "Person", "Bus", "Bridge", "Road Divider", "Tempo", "Traffic Sign Board", "Flag", "Crane",
        "Cycle", "Dog", "Truck", "Glove", "Overbridge", "Manhole", "Bus Stop", "Barricade",
        "Petrol Pump", "Ambulance", "Goat", "Cart"
    ]
}

# Save the configuration to a temporary file
config_file_path = '/content/drive/MyDrive/UAV/config.yaml'
with open(config_file_path, 'w') as file:
    yaml.dump(data_config, file)

print(f"Config file saved at {config_file_path}")

# Upload the weights file 'last.pt' to continue training
print("Please upload the weights file.")
uploaded = files.upload()  # Upload the file

# Print the names of the uploaded files for debugging
print("Uploaded files:", uploaded.keys())

# Ensure the uploaded file is correctly handled
uploaded_files = list(uploaded.keys())
if len(uploaded_files) == 0:
    raise FileNotFoundError("No file uploaded. Please upload the correct weights file.")
uploaded_file_name = uploaded_files[0]

# Handle file names like 'last (8).pt', 'last (9).pt', etc.
if 'last.pt' not in uploaded_file_name:
    # Rename the uploaded file to 'last.pt' if it contains 'last' and a number
    new_file_name = 'last.pt'
    os.rename(f'/content/{uploaded_file_name}', f'/content/{new_file_name}')
    uploaded_file_name = new_file_name
    print(f"Renamed the uploaded file to: {uploaded_file_name}")

# Path to the uploaded weights file
last_weights_path = f'/content/{uploaded_file_name}'
print(f"Using uploaded file: {uploaded_file_name}")

# Load the model from the uploaded weights
model = YOLO(last_weights_path)

# Resume training for 5 epochs or until 50 epochs total
total_epochs = 50
current_epoch = model.ckpt['epoch'] + 1  # Add 1 to the epoch because YOLOv8 starts from 0
remaining_epochs = total_epochs - current_epoch
epochs_to_train = min(5, remaining_epochs)  # Train for 5 or fewer epochs if near the target

if remaining_epochs > 0:
    print(f"Training for {epochs_to_train} epochs (Current Epoch: {current_epoch}).")
    model.train(data=config_file_path, epochs=current_epoch + epochs_to_train, batch=5)

    # Update the current epoch after training
    current_epoch += epochs_to_train

    # Save the updated weights
    new_weights_path = f'/content/drive/MyDrive/updated_weights_epoch_{current_epoch}.pt'
    model.save(new_weights_path)
    print(f"Updated weights saved at {new_weights_path}.")
else:
    print("Training is already complete. Total 50 epochs reached!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config file saved at /content/drive/MyDrive/UAV/config.yaml
Please upload the weights file.


Saving last.pt to last (1).pt
Uploaded files: dict_keys(['last (1).pt'])
Renamed the uploaded file to: last.pt
Using uploaded file: last.pt
Training for 5 epochs (Current Epoch: 0).
Ultralytics 8.3.36 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=/content/last.pt, data=/content/drive/MyDrive/UAV/config.yaml, epochs=5, time=None, patience=100, batch=5, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment

100%|██████████| 755k/755k [00:00<00:00, 36.2MB/s]



                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

100%|██████████| 5.35M/5.35M [00:00<00:00, 180MB/s]


...

                   all        566       3078      0.548      0.491      0.492      0.275

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        5/5     0.807G      1.229      1.051      1.111         22        640: 100%|██████████| 373/373 [05:13<00:00,  1.19it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 57/57 [01:32<00:00,  1.61s/it]


                   all        566       3078      0.621      0.481       0.52       0.29

5 epochs completed in 0.575 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 6.2MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.36 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
Model summary (fused): 168 layers, 3,012,473 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 57/57 [01:28<00:00,  1.56s/it]


                   all        566       3078      0.621      0.483       0.52       0.29
        Traffic Signal        160        277      0.736      0.675      0.729      0.385
             Lamp Post        112        213      0.528      0.526      0.485      0.222
        Zebra Crossing         39         62      0.368      0.242      0.289      0.159
                  Bike        300        634      0.834      0.793      0.833      0.419
                   Car        234        461      0.775      0.792       0.83      0.502
               Rikshaw        143        213      0.664      0.728      0.719      0.453
            Tyre Works         28         53       0.53      0.585      0.591      0.331
                  Tree         98        134      0.612      0.642      0.651      0.395
               Tractor         15         18      0.585      0.315      0.513      0.264
                Cattle         16         39      0.658       0.79      0.698      0.419
            Vegetatio

In [8]:
import shutil
from google.colab import files

# Specify the source directory and the target ZIP file path
source_dir = "/content/runs"
output_zip = "/content/runs.zip"

# Create a ZIP file from the source directory
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', source_dir)

# Download the ZIP file to the local system
files.download(output_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>